<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2025/images/logo.png?raw=true" alt="2025年度ゲノム情報解析入門" height="100px" align="middle">

<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>

# ゲノムアセンブル
　近年、シロイヌナズナやイネなどのようなモデル生物だけでなく、さまざまな生物でゲノム配列の整備（ゲノム解読）が進められています。解読されたゲノム配列は、進化学的研究や育種学的研究など、幅広い研究の基盤情報になります。

　次世代シーケンシング（ショートリードシーケンシング）や第三世代シーケンシング（ロングリードシーケンシング）をおこなうと、元々のゲノム配列に由来する、短い塩基配列（**リード、Read**）を多数得られます。ショートリードシーケンシングでは50-300 bpのリードを取得でき、ロングリードシーケンシングでは数kbから数十kb（ときには数百kb~数Mb）のリードを取得できます。ゲノム解読は、それらのリードをつなぎ、元のゲノム配列を復元する試みです。その復元処理の最初のステップは、 **ゲノムアセンブル（Genome assemble）** と呼ばれています。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/assemble_overview.png?raw=true" height="100px" align="middle">

　　今回の実習では、[約250 kbのゲノム配列](https://raw.githubusercontent.com/CropEvol/lecture/refs/heads/master/textbook_2025/dataset/sample_genome.fa) から仮想的に得たショートリードデータやロングリードデータを使って、それぞれからどのようなゲノム配列を復元できるのかを見ていきます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/short_and_long_reads.png?raw=true" height="100px" align="middle">



### サンプルデータのダウンロード

---



In [ ]:
### サンプルデータのダウンロード
!wget -q -O - https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/illumina_read1.fastq.gz | gzip -dc > illumina_read1.fastq
!wget -q -O - https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/illumina_read2.fastq.gz | gzip -dc > illumina_read2.fastq
!wget -q -O - https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/nanopore.fastq.gz | gzip -dc > nanopore.fastq
!wget -q -O - https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/sample_genome.fa > sample_genome.fa

### ソフトウェアのインストール

In [ ]:
### ソフトウェアのインストール
!wget -q -O 'assemble_softwares.sh' https://raw.githubusercontent.com/CropEvol/lecture/master/textbook_2025/scripts/assemble_softwares.sh
!bash assemble_softwares.sh

### 今回の勉強内容

1. ショートリードを使ったゲノムアセンブル（アセンブルのしくみ）

1. ロングリードを使ったゲノムアセンブル（アセンブリの正しさを確かめる）

1. ゲノム配列を改良する

1. 発展課題: リードの量を変えてアセンブルする


---
## 1. ショートリードを使ったゲノムアセンブル

　ショートリードのゲノムアセンブルをおこない、ゲノムの再構築を試みてみましょう。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/short_read_assemble.png?raw=true" height="100px" align="middle">

### 1-1. ショートリードを確認する
　最初に、ゲノムアセンブルに使用するショートリードデータを確認してみましょう。DNA断片の両端をシーケンシングした **ペアエンド（Paired-end）** という形のリードデータを使用します。ペアエンドのデータは、通常2つのファイルに分かれており、それらを1組のデータとして同時に扱います。

使用するショートリード:
- Read1: /content/illumina_read1.fastq
- Read2: /content/illumina_read2.fastq
- リードの量: ゲノムサイズの約20倍
- リードの長さ: 各80 bp

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/paired_end_reads.png?raw=true" height="200px" align="middle">

#### 復習：リードのデータファイル形式（FASTQ）


　リードのデータは**FASTQ**と呼ばれる形式でファイルに保存されています。FASTQ形式では、4行で1つのリードデータの情報を表しています。

```
@リードの名前
TGAAGCAGT..............(リードの塩基配列)
+
F1GCFCGGE..............(各塩基のクオリティ)
```

- 第1行（@からはじまる行）: リードの名前
- 第2行: リードの塩基配列
- 第3行（+からはじまる行）: 第2行と第4行の区切りのための行
- 第4行: 各塩基配列のクオリティ（ASCIIコードで書かれている）。1文字が何らかのクオリティスコア（数値）を示している。
 - 参考: https://bi.biopapyrus.jp/rnaseq/qc/fastq-quality-score.html



#### seqkitを使ったリードの統計情報確認

 　[seqkit](https://bioinf.shenwei.me/seqkit/)というソフトウェアを使って、ペアエンドのうちRead1（ファイル名: `illumina_read1.fastq`）のリードの数やリードの長さなどの情報を確認してみましょう。

```sh
# リードの統計情報の確認
seqkit stat 塩基配列データ
```

In [ ]:
%%bash
seqkit stat illumina_read1.fastq

　下のコードセルに追記して、もう一方のリード（Read2のファイル名: `illumina_read2.fastq`）の情報も確認してください。

In [ ]:
%%bash
seqkit stat

### 1-2. アセンブルのしくみを体験する

　SPAdesを実行する前に、ショートリードのアセンブルソフトウェアが中で何をしているのかを、小さな例で体験してみましょう。

　多くのショートリード用のアセンブルソフトウェアは、リードを長さ **k** の短い断片（**k-mer**）に分解し、k-mer同士の重なり（k−1塩基）をたどってつなぐことで、元の配列を復元します。この方法は **de Bruijn（ド・ブラン）グラフ法** と呼ばれています。

```
リード:  ATGGCGTA       k=5 のk-mer:  ATGGC
          TGGCGTAC                      TGGCG   ← 前のk-merと4塩基（k−1）重なる
           GGCGTACG                      GGCGT
             ...                           ...
```

　つなぎ方が1通りに決まるところはそのまま伸ばし、つなぎ方が複数ある分かれ道に来たら、そこでコンティグを切ります。

　まず、次のコードセルを実行して、アセンブルをおこなう小さなプログラムを準備してください（中身を理解する必要はありません）。

In [ ]:
## 小さなアセンブルプログラム（de Bruijnグラフ法の簡易版）
from collections import defaultdict

def make_reads(genome, read_len):
    ## ゲノム配列から、1塩基ずつずらしながらリードを切り出す（エラーなしのリード）
    return [genome[i:i+read_len] for i in range(len(genome) - read_len + 1)]

def assemble(reads, k):
    ## リードをk-merに分解する
    kmers = set()
    for r in reads:
        for i in range(len(r) - k + 1):
            kmers.add(r[i:i+k])
    ## k-merの重なり（k-1塩基）でグラフをつくる
    out_edges = defaultdict(set); in_edges = defaultdict(set)
    for km in kmers:
        out_edges[km[:-1]].add(km[1:]); in_edges[km[1:]].add(km[:-1])
    nodes = set(out_edges) | set(in_edges)
    ## つなぎ方が1通りでない場所（分かれ道・合流点・端）
    def is_junction(n):
        return not (len(out_edges[n]) == 1 and len(in_edges[n]) == 1)
    ## 分かれ道から次の分かれ道まで、1本道をたどってコンティグにする
    contigs = []
    for n in nodes:
        if is_junction(n):
            for nxt in out_edges[n]:
                path = [n, nxt]
                while not is_junction(path[-1]):
                    path.append(next(iter(out_edges[path[-1]])))
                contigs.append(path[0] + "".join(p[-1] for p in path[1:]))
    return sorted(contigs, key=len, reverse=True)

def show(genome, read_len, k):
    contigs = assemble(make_reads(genome, read_len), k)
    print(f"ゲノム長 {len(genome)} bp, リード長 {read_len} bp, k={k} → コンティグ {len(contigs)} 本")
    for c in contigs:
        print("  ", c, f"({len(c)} bp)")
    if len(contigs) == 1:
        print("   元の配列を復元できた？:", contigs[0] == genome)

#### 例1: リピートのないゲノム

　47塩基の小さなゲノムから、長さ15塩基のリードを作ってアセンブルします。k=7でアセンブルすると、元の配列を1本のコンティグとして復元できます。

In [ ]:
genome1 = "ATGGCGTACGCTTAGGCTAACCGGATCCTTGAGCAATCGGTACCATG"
show(genome1, read_len=15, k=7)

#### 練習1
　上のコードセルの `k=7` を `k=5` に変えて実行してください。コンティグは何本になりましたか？ なぜ分かれてしまったのか考えてみましょう（ヒント: 短いk-merは、ゲノム中に偶然2回現れることがあります）。

#### 例2: リピートのあるゲノム

　次は、ゲノムの中に同じ10塩基の配列（`CGATTGCAGT`）が2回現れる場合です。

In [ ]:
repeat = "CGATTGCAGT"   # 10塩基のリピート
genome2 = "ATGGCGTA" + repeat + "CTTAGGCTAACC" + repeat + "GGATCCTTGAGC"
show(genome2, read_len=15, k=9)

#### 練習2
　上のコードセルの `k` を 9 → 11 → 13 と大きくしながら実行し、コンティグの本数がどう変わるか確認してください。kがどのくらいの長さになると、元の配列を1本に復元できましたか？

#### 例3: リード長よりも長いリピート

　最後に、リピートの長さを20塩基にしてみます。リード長は15塩基のままです。

In [ ]:
repeat = "CGATTGCAGTTACCGATAGC"   # 20塩基のリピート
genome3 = "ATGGCGTA" + repeat + "CTTAGGCTAACC" + repeat + "GGATCCTTGAGC"
show(genome3, read_len=15, k=15)

#### 練習3
1. k-merはリードから切り出すので、kはリード長（15）より大きくできません。k=15でも、元の配列を復元できましたか？
1. 上のコードセルを、`read_len=30, k=25` に書き換えて実行してください。結果はどう変わりましたか？

　この実習の結果から、次のことがわかります。

- リピートの長さより長いk-merを使えば、リピートの前後を区別してつなげられる
- しかし、kはリード長より長くできないため、**リード長より長いリピートは、ショートリードだけでは原理的に区別できない**
- リードを長くすれば（＝ロングリードを使えば）、長いリピートも乗り越えられる

　実際のSPAdesは、複数のkを組み合わせたり、ペアエンドの情報を使ったりして、もっと工夫してつないでいます。それでも、この限界があることに変わりはありません。では、実際のデータでアセンブルをおこなってみましょう。

### 1-3. ゲノムアセンブル

　今回、[SPAdes](https://github.com/ablab/spades)というショートリードのゲノムアセンブルをおこなえるソフトウェアを使って、ショートリードを使ったゲノム配列の復元を試みてみます。

　次のコードセルのプログラムで、ペアエンドショートリードのゲノムアセンブルをおこなってみてください。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/genome_assemble.png?raw=true" height="250px" align="middle">

In [ ]:
%%bash
## ショートリードのアセンブル
## [入力] -1 ショートリードREAD1ファイル -2 ショートリードREAD2ファイル
## [出力] -o 出力フォルダ名
spades.py -1 illumina_read1.fastq -2 illumina_read2.fastq -o spades_output

### 1-4. ショートリードゲノムアセンブルの結果


　アセンブルで得られたゲノム配列（**アセンブリ、Assembly**）を確認してみましょう。Colabサーバー上の次のファイルに保存されています。

/content/spades_output/scaffolds.fasta

　アセンブリは、1本ではなく、複数本の配列に分かれており、元のゲノム配列を完全に再現できていません。アセンブル後の複数本に分かれた配列は **コンティグ（Contig）** と呼ばれています。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/contigs.png?raw=true" height="100px" align="middle">


　seqkitを使って、コンティグの配列数や長さの情報をみてみましょう。



In [ ]:
%%bash
## アセンブルの統計情報
## オプション「-a」を追加すると、より多くの情報を得られます
seqkit stat -a spades_output/scaffolds.fasta

#### アセンブルを評価する指標のひとつ「N50」

　再構築したゲノム配列（アセンブリ）を評価する方法をいくつかあります。そのうちのひとつが **N50** と呼ばれる配列の長さを使った指標です。N50は、コンティグを長いものから並べて、全長の50%をカバーするときのコンティグの長さです。ゲノムの半分をその長さ以上のコンティグで再構築できていることを示している指標です。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/about_n50.png?raw=true" height="150px" align="middle">


　ショートリードで、元のゲノム配列（約250 kb）のほぼ全長を復元することができました。しかし、100本以上の配列に分かれており、それぞれの配列の長さも元のゲノム配列からするとまだ短い状態です。

---
## 2. ロングリードを使ったゲノムアセンブル

　ショートリードシーケンシングは、50-300 bpの短い塩基配列を多量に得るシーケンシングです。近年、非常に長い塩基配列を得るシーケンシング技術（**ロングリードシーケンシング、Long read sequencing**）が発達してきました。Oxford Nanopore Technologies社のナノポアシーケンシングや、Pacific Biosciences社のPacBioシーケンシングがそれにあたります。

　現在では、ロングリードを使ったゲノムアセンブルが最もよくおこなわれています。ここでは、ナノポアシーケンシングによるロングリードからどのようなゲノム配列を復元できるかを見ていきましょう。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/long_read_assemble.png?raw=true" height="100px" align="middle">

### 2-1. ロングリードを確認する

　ロングリードのシーケンスデータを見てみましょう。

使用するロングリード:
- ロングリード: /content/nanopore.fastq
- リードの量: ゲノムサイズの約8倍
- リードの長さ: 平均の長さが約5000 bp

　seqkitを使って、ロングリードの統計情報も確認してみましょう。

In [ ]:
%%bash
seqkit stat nanopore.fastq

### 2-2. ゲノムアセンブル
　ロングリードを使ったアセンブルでは、ロングリード用のソフトウェア[Flye](https://github.com/fenderglass/Flye)を使って、ゲノムアセンブルを試みてみましょう。



In [ ]:
%%bash
## ロングリードを使ったアセンブル
flye --scaffold --nano-raw nanopore.fastq --out-dir flye_output

### 2-3. ロングリードゲノムアセンブルの結果


　アセンブリは、次のファイルに保存されます。

/content/flye_output/assembly.fasta

　seqkitで、アセンブリの統計情報を確認してみましょう。

In [ ]:
%%bash
seqkit stat -a flye_output/assembly.fasta

#### Flyeのアセンブル結果の詳細を見る

　Flyeは、各コンティグの情報を `assembly_info.txt` というファイルにまとめて出力します。中身を見てみましょう。

In [ ]:
%%bash
cat flye_output/assembly_info.txt

　主な列の意味は次のとおりです。

|列名|意味|
|:--|:--|
|`#seq_name`|コンティグの名前|
|`length`|コンティグの長さ|
|`cov.`|コンティグに重なっているリードの平均本数（カバレッジ）|
|`circ.`|環状のコンティグかどうか（Y: 環状、N: 線状）|
|`repeat`|リピートと判定されたコンティグかどうか|

　リードはゲノム全体からほぼ均等に得られるので、ゲノム中に1か所しかない配列のカバレッジは、どのコンティグでも同じくらいになるはずです。**カバレッジが他のコンティグの約2倍のコンティグがあったら、それは何を意味すると考えられるでしょうか？**

　ここまでの結果を、下の表にまとめてみましょう（このテキストセルをダブルクリックすると編集できます）。seqkitの出力の `num_seqs` が配列数、`sum_len` が全長、`N50` がN50です。

|アセンブリ|配列数|全長|N50|
|:--|--:|--:|--:|
|ショートリードのアセンブリ| | | |
|ロングリードのアセンブリ| | | |

　ロングリードを使ったゲノムアセンブルにより、ショートリードのときの結果よりも全長がやや長くなりました。全長がやや長くなっただけではなく、配列数が大幅に少なくなり、N50の値もかなり向上しました。

<small>※ 参考: このテキストの作成時の結果は、ショートリードのアセンブリが配列数140本・N50 11,812 bp、ロングリードのアセンブリが配列数8本・N50 56,643 bpでした。ソフトウェアのバージョンなどによって、数値が少し変わることがあります。</small>

### 2-4. ショートリードとロングリードのゲノムアセンブル結果を比べる

　元のゲノム配列に対して、ショートリードとロングリードのアセンブリを並べてみて、それぞれのアセンブリの特徴を見てみましょう。ある塩基配列に別の塩基配列を並べる操作を **アライメント（Alignment）** と言います。
- ショートリードアセンブリを元のゲノム配列にアライメントする
- ロングリードアセンブリを元のゲノム配列にアライメントする

In [ ]:
%%bash
## ショートリードアセンブリを元のゲノム配列にアライメント
OUTPUT="ShortReadAssembly"
REF="sample_genome.fa"
CONTIG="spades_output/scaffolds.fasta"
bash Alignment_mm2.sh $REF $OUTPUT $CONTIG

## ロングリードアセンブリを元のゲノム配列にアライメント
OUTPUT="LongReadAssembly"
REF="sample_genome.fa"
CONTIG="flye_output/assembly.fasta"
bash Alignment_mm2.sh $REF $OUTPUT $CONTIG

In [ ]:
## アライメント結果をみる
fa   = "sample_genome.fa"
bam1 = "ShortReadAssembly.bam"
bam2 = "LongReadAssembly.bam"

## ---- 以下、変更不要 -----
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
bam_common_info = { "displayMode":"SQUISHED", "viewAsPairs":True, "height":100, "showCoverage":False }
igv_notebook.init()
ref = RefTrack({ "fastaPath": fa, "indexPath": fa+".fai", "id": fa })
aln1 = BamTrack({ "name": bam1, "path": bam1, "height":120, "showCoverage":False })
aln2 = BamTrack({ "name": bam2, "path": bam2, "height":120, "showCoverage":False })
b = igv_notebook.Browser(ref)
b.load_track(aln1)
b.load_track(aln2)


　ショートリードは、長さは短い（今回は80 bp）ものの、塩基の読み間違い（シーケンシングエラー）が少ないリードです。そのため、ショートリードからは、正確性の高いアセンブリを構築できます。

　しかし、生物のゲノムには、同じ塩基パターンの繰り返し（リピート）や重複などが多く含まれています。ショートリードは、リード長よりも長い塩基パターンの繰り返し等に弱く、ショートリードだけでゲノムを再現するのはほぼ不可能です。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/short_reads_and_repeat.png?raw=true" height="250px" align="middle">

　一方で、ロングリードは、シーケンシングエラー率が高く、構築されたコンティグの塩基が正確ではない場合があります。しかし、ショートリードの場合とは異なり、ロングリードを使ったゲノムアセンブリは、非常に長いコンティグを生成できます。そのため、リピート配列を多く含む生物のゲノムアセンブリにおいて、ロングリードは主力的な役割を果たしています。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/long_read_assemble2.png?raw=true" height="250px" align="middle">

### 2-5. コンティグの切れ目はリピートなのか確かめる

　上の説明では、ショートリードのアセンブリがリピートで途切れると書きました。1-2 の小さな例でも、そのことを確認しました。今回の実際のデータでも本当にそうなっているのか、確かめてみましょう。

　次のコードセルでは、まず元のゲノム配列の中で **2回以上現れる80塩基の配列（リード長と同じ長さの繰り返し）** をリピートとして探します。逆向きの相補鎖で現れる場合も含めます。次に、ショートリードのコンティグの端が、そのリピートの近く（150塩基以内）にあるかを調べます。比較のため、ゲノム上のランダムな位置についても同じ計算をします。

In [ ]:
from collections import Counter
import random, subprocess
from Bio import SeqIO

record = next(SeqIO.parse("sample_genome.fa", "fasta"))
genome = str(record.seq).upper()

## (1) 2回以上現れる80塩基の配列（リピート）を探す
k = 80
comp = str.maketrans("ACGTN", "TGCAN")
def canonical(s):
    ## 配列とその逆相補鎖のうち、辞書順で小さい方を代表にする（向きの違いを同一視する）
    return min(s, s[::-1].translate(comp))
kmers = [canonical(genome[i:i+k]) for i in range(len(genome) - k + 1)]
count = Counter(kmers)
is_repeat = [False] * len(genome)
for i, km in enumerate(kmers):
    if count[km] >= 2 and "N" not in km:
        for j in range(i, i + k):
            is_repeat[j] = True
print(f"ゲノム中のリピートの割合: {sum(is_repeat)/len(genome)*100:.1f}%")

## リピートの位置をBED形式のファイルに書き出す（後でIGVで表示する）
regions, start = [], None
for i, r in enumerate(is_repeat + [False]):
    if r and start is None: start = i
    if not r and start is not None: regions.append((start, i)); start = None
with open("repeats.bed", "w") as f:
    for s, e in regions:
        f.write(f"{record.id}\t{s}\t{e}\trepeat\n")

## (2) ショートリードのコンティグを元のゲノムに並べ、コンティグの端の位置を調べる
subprocess.run("minimap2 -c -x asm5 sample_genome.fa spades_output/scaffolds.fasta > ShortReadAssembly.paf 2>/dev/null", shell=True)
ends = []
for line in open("ShortReadAssembly.paf"):
    c = line.split("\t")
    if "tp:A:P" in line:              # 主なアライメントだけを使う
        ends += [int(c[7]), int(c[8])]  # 元のゲノム上の開始・終了位置

## (3) コンティグの端と、ランダムな位置で、近くにリピートがある割合を比べる
w = 150
def near_repeat(p):
    return any(is_repeat[max(0, p - w):p + w])
random.seed(1)
random_pos = [random.randrange(len(genome)) for _ in range(len(ends) * 20)]
print(f"コンティグの端 {len(ends)} か所のうち、近くにリピートがある割合: {sum(map(near_repeat, ends))/len(ends)*100:.1f}%")
print(f"ランダムな位置で、近くにリピートがある割合        : {sum(map(near_repeat, random_pos))/len(random_pos)*100:.1f}%")

　コンティグの端と、ランダムな位置とで、割合に大きな差が出たでしょうか？ 次のコードセルで、リピートの位置とショートリードのアセンブリをIGVで並べて見てみましょう。

　ここでは例として、2.8 kb付近でコンティグが途切れている領域を表示します。`locus` を書き換えたり、画面を移動したりして、ほかの場所も見てみてください。

In [ ]:
## アライメント結果をみる（リピートの位置とショートリードのアセンブリ）
fa    = "sample_genome.fa"
bam1  = "ShortReadAssembly.bam"
locus = "original:1,500-5,500"   # 最初に表示する領域
## ---- 以下、変更不要 -----
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
igv_notebook.init()
ref  = RefTrack({ "fastaPath": fa, "indexPath": fa+".fai", "id": fa }, locus=locus)
rep  = AnnotationTrack({ "name": "repeats (80 bp)", "path": "repeats.bed", "format": "bed" })
aln1 = BamTrack({ "name": bam1, "path": bam1, "height":200, "showCoverage":False })
b = igv_notebook.Browser(ref)
b.load_track(rep)
b.load_track(aln1)

### 2-6. アセンブリの正しさを確かめる（ドットプロット）

　ここまで、N50などの「長さ」の指標でアセンブリを比べてきました。しかし、長いコンティグが得られても、**つなぎ方が正しいとは限りません**。本来は離れた場所にある配列を、間違ってつないでしまうこと（**ミスアセンブリ、Misassembly**）があるからです。

　アセンブリの正しさを確かめる方法のひとつが **ドットプロット** です。横軸に元のゲノム配列、縦軸にアセンブリのコンティグをとり、対応する部分に線を引きます。アセンブリが正しければ、線は左下から右上への**1本の斜めの直線**に並びます。

<small>※ 今回は元のゲノム配列がわかっているので、このような比較ができます。実際のゲノム解読では正解がないため、近縁種のゲノムと比べたり、別の種類のデータで検証したりします。</small>

In [ ]:
import subprocess
import matplotlib.pyplot as plt

def run_minimap2(contig_file, paf_file):
    ## コンティグを元のゲノムに並べ、結果（PAF形式）を読み込む
    subprocess.run(f"minimap2 -c -x asm5 sample_genome.fa {contig_file} > {paf_file} 2>/dev/null", shell=True)
    hits = []
    for line in open(paf_file):
        c = line.split("\t")
        if "tp:A:P" not in line: continue
        hits.append(dict(contig=c[0], clen=int(c[1]), qs=int(c[2]), qe=int(c[3]), strand=c[4],
                         ts=int(c[7]), te=int(c[8]), alen=int(c[10])))
    return hits

def dotplot(ax, hits, title):
    ## ほとんどが逆向き（-）に並んだコンティグは、向きをそろえて表示する
    hits = [dict(h) for h in hits]
    for c in {h["contig"] for h in hits}:
        hs = [h for h in hits if h["contig"] == c]
        if sum(h["alen"] for h in hs if h["strand"] == "-") > sum(h["alen"] for h in hs if h["strand"] == "+"):
            for h in hs:
                h["qs"], h["qe"] = h["clen"] - h["qe"], h["clen"] - h["qs"]
                h["strand"] = "+" if h["strand"] == "-" else "-"
    ## コンティグを、元のゲノム上の位置の順に縦に積み上げる
    first_pos = {}
    for h in hits:
        if h["contig"] not in first_pos or h["alen"] > first_pos[h["contig"]][1]:
            first_pos[h["contig"]] = (h["ts"], h["alen"])
    order = sorted(first_pos, key=lambda c: first_pos[c][0])
    clen = {h["contig"]: h["clen"] for h in hits}
    offset, total = {}, 0
    for c in order:
        offset[c] = total; total += clen[c]
        ax.axhline(offset[c]/1000, color="lightgray", lw=0.5)   # コンティグの境目
    for h in hits:
        y0, y1 = offset[h["contig"]] + h["qs"], offset[h["contig"]] + h["qe"]
        if h["strand"] == "-": y0, y1 = y1, y0
        ax.plot([h["ts"]/1000, h["te"]/1000], [y0/1000, y1/1000],
                color="tab:blue" if h["strand"] == "+" else "tab:red", lw=1.5)
    ax.set_xlabel("Original genome (kb)"); ax.set_ylabel("Assembly contigs (kb)"); ax.set_title(title)

short_hits = run_minimap2("spades_output/scaffolds.fasta", "ShortReadAssembly.paf")
long_hits  = run_minimap2("flye_output/assembly.fasta",   "LongReadAssembly.paf")
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
dotplot(axes[0], short_hits, "Short-read assembly (SPAdes)")
dotplot(axes[1], long_hits,  "Long-read assembly (Flye)")
plt.tight_layout()
plt.show()

##### グラフの見方
- 横軸: 元のゲノム配列上の位置（kb）
- 縦軸: アセンブリのコンティグを積み上げたもの（kb）。灰色の横線がコンティグの境目です
- 青線: コンティグと元のゲノムが同じ向きで対応している部分、赤線: 逆向きで対応している部分

　ロングリードのアセンブリでは、線が途中でずれたり、ひとつのコンティグ（灰色の線に挟まれた範囲）の中で、元のゲノムの離れた場所に飛んでいたりする部分はありませんか？ 次のコードセルで、ロングリードのコンティグごとに、どの部分が元のゲノムのどこに対応しているかを確認しましょう。

In [ ]:
for c in sorted({h["contig"] for h in long_hits}):
    segs = sorted([h for h in long_hits if h["contig"] == c], key=lambda h: h["qs"])
    print(f"{c}（{segs[0]['clen']:,} bp）")
    for h in segs:
        print(f"   コンティグ上 {h['qs']:>7,}-{h['qe']:>7,}  →  元のゲノム上 {h['ts']:>7,}-{h['te']:>7,} ({h['strand']})")

### 考えてみよう
1. ロングリードのコンティグの中で、元のゲノムの**離れた場所どうしがつながっている**コンティグはどれですか？（ヒント: 1本のコンティグの中で、「元のゲノム上」の位置が順番に並んでいないもの）
1. 元のゲノムの**同じ場所に、2本のコンティグが重なって並んでいる**ところはありますか？
1. ショートリードのアセンブリでは、このようなつなぎ間違いは見られましたか？
1. 「N50が大きい＝良いアセンブリ」と言えるでしょうか？ アセンブリを評価するには、どのような指標が必要だと思いますか？

<small>※ 実際の研究では、[QUAST](https://github.com/ablab/quast) などのソフトウェアを使って、つなぎ間違いの数や塩基の誤りの数などを、長さの指標と合わせて評価します。Flyeは実行環境によって結果が少し変わることがあるので、上のつなぎ間違いの場所は、人によって異なる場合があります。</small>

---
## 3. ゲノム配列を改良する

　リード数が多く、塩基情報が正確なショートリードと、長いコンティグを生成できるロングリードを組み合わせることで、より良いゲノム配列を得ることができます。

　先ほど得られたロングリードのゲノム配列（コンティグ）には、塩基情報が不正確そうな箇所がみられます。ここでは、ショートリードでその不正確性の補正を試みてみましょう。この補正処理は **ポリッシング（Polishing）** と呼ばれています。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/polishing.png?raw=true" height="220px" align="middle">

### 3-0. ソフトウェアのインストール

In [ ]:
## 必要なソフトウェアをインストールする
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q -O tools/pilon-1.24.jar https://github.com/broadinstitute/pilon/releases/download/v1.24/pilon-1.24.jar
!wget -q -O Polishing.sh https://github.com/CropEvol/lecture/raw/master/textbook_2025/scripts/Polishing_java.sh

#### ソフトウェアのインストール（代替手段）
　上の3-0のコードセルでソフトウェアをインストールできない場合、こちらをお試しください。

<small>※ このコードセルを実行すると、完了時に画面下に「セッションがクラッシュしました。」の表示が出ますが、ほとんどの場合、正常にソフトウェアがインストールされます。</small>

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install_miniforge()
!conda install -q -y bioconda::bwa
!conda install -q -y bioconda::pilon
!wget -q -O Polishing.sh     https://github.com/CropEvol/lecture/raw/master/textbook_2025/scripts/Polishing.sh

### 3-1. ショートリードを使った補正

In [ ]:
%%bash
CONTIG="flye_output/assembly.fasta"
OUTPUT="assembly"
READ1="illumina_read1.fastq"
READ2="illumina_read2.fastq"

bash Polishing.sh $CONTIG $OUTPUT $READ1 $READ2

### 3-2. 補正後のゲノム配列を確認する

　補正後のゲノム配列にショートリードをアライメントし、その結果をIGVで確認してみましょう。

In [ ]:
%%bash
## 補正前のアセンブリをオリジナルのゲノム配列にアライメント
REF="sample_genome.fa"
OUTPUT1="LongReadAssembly"
CONTIG1="flye_output/assembly.fasta"
bash Alignment_mm2.sh $REF $OUTPUT1 $CONTIG1

## 補正後のアセンブリをオリジナルのゲノム配列にアライメント
OUTPUT2="LongReadAssembly_polished"
CONTIG2="assembly_polished.fasta"
bash Alignment_mm2.sh $REF $OUTPUT2 $CONTIG2

In [ ]:
## アライメント結果をみる
fa   = "sample_genome.fa"
bam1 = "LongReadAssembly.bam"
bam2 = "LongReadAssembly_polished.bam"

## ---- 以下、変更不要 -----
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
bam_common_info = { "displayMode":"SQUISHED", "viewAsPairs":True, "height":100, "showCoverage":False }
igv_notebook.init()
ref = RefTrack({ "fastaPath": fa, "indexPath": fa+".fai", "id": fa })
aln1 = BamTrack({ "name": bam1, "path": bam1, "height":120, "showCoverage":False })
aln2 = BamTrack({ "name": bam2, "path": bam2, "height":120, "showCoverage":False })
b = igv_notebook.Browser(ref)
b.load_track(aln1)
b.load_track(aln2)


### 3-3. 補正の効果を数で確かめる

　IGVで見ると、補正によって塩基の誤りが減ったように見えます。では、実際にいくつ減ったのでしょうか？ 補正前と補正後のアセンブリを元のゲノム配列と比べて、コンティグごとに **置換**（違う塩基になっている）、**挿入**（余分な塩基がある）、**欠失**（塩基が抜けている）の箇所数を数えてみましょう。

In [ ]:
import re, subprocess
import pandas as pd

def count_errors(contig_file):
    ## コンティグを元のゲノムに並べ、違いの情報（csタグ）を出力する
    paf = subprocess.run(f"minimap2 -c --cs -x asm5 sample_genome.fa {contig_file}",
                         shell=True, capture_output=True, text=True).stdout
    rows = {}
    for line in paf.splitlines():
        if "tp:A:P" not in line: continue
        name = line.split("\t")[0].replace("_pilon", "")   # 補正後のコンティグ名の「_pilon」を除く
        cs = re.search(r"cs:Z:(\S+)", line).group(1)
        r = rows.setdefault(name, {"置換": 0, "挿入": 0, "欠失": 0})
        ## csタグ: *ag = 置換, +acg = 挿入, -acg = 欠失
        for op in re.findall(r"\*[a-z]{2}|\+[a-z]+|-[a-z]+", cs):
            if op[0] == "*":   r["置換"] += 1
            elif op[0] == "+": r["挿入"] += 1
            else:              r["欠失"] += 1
    return pd.DataFrame(rows).T

before = count_errors("flye_output/assembly.fasta")
after  = count_errors("assembly_polished.fasta")
table = pd.concat({"補正前": before, "補正後": after}, axis=1).sort_index()
table.loc["合計"] = table.sum()
table

### 考えてみよう
1. 補正によって、置換・挿入・欠失のどれが一番大きく減りましたか？ ナノポアのロングリードには、同じ塩基が連続する部分（例: `AAAAAA`）で塩基数を読み間違えやすいという特徴があります。この特徴と結果を結びつけて考えてみましょう。
1. 補正後も誤りが多く残っているコンティグはどれですか？ そのコンティグが元のゲノムのどこに並んでいるか、2-6 のドットプロットとコンティグごとの対応表で確認してみましょう。
1. ポリッシングでは、ショートリードをコンティグに並べ、多くのリードが示す塩基に修正します。2. の答えのコンティグで補正がうまくいかなかったのはなぜでしょうか？（ヒント: 同じ配列が2本のコンティグにあると、ショートリードはどちらに並ぶでしょうか？）

　今回、ロングリードのアセンブル結果をショートリードで補正しました。そのほかにも、ロングリードをショートリードで補正し、エラーの少ないロングリードにした後にアセンブルをおこなう方法や、ショートリードのアセンブリをロングリードでより長くする方法などがあります。

---
## 発展課題（時間が余った人向け）: リードの量を変えてアセンブルする

　今回のショートリードは、ゲノムサイズの約20倍の量がありました。リードの量（**カバレッジ**）を減らすと、アセンブルの結果はどう変わるでしょうか？

　次のコードセルでは、`seqkit sample` でショートリードの一部をランダムに取り出し、そのリードでSPAdesのアセンブルをおこないます。`-p 0.25` は、全体の25%のリード（約5倍のカバレッジ）を取り出すという意味です。Read1とRead2で同じ乱数の種（`-s 11`）を使うと、ペアの組み合わせを保ったまま取り出せます。

　周りの人と分担して、`P` の値を `0.5`（約10倍）、`0.25`（約5倍）、`0.1`（約2倍）などに変えて実行し、結果を比べてみましょう。

In [ ]:
%%bash
P=0.25   # 取り出すリードの割合（0.5, 0.25, 0.1 などに変えてみる）

seqkit sample -p $P -s 11 illumina_read1.fastq > sub_read1.fastq 2>/dev/null
seqkit sample -p $P -s 11 illumina_read2.fastq > sub_read2.fastq 2>/dev/null
seqkit stat sub_read1.fastq sub_read2.fastq

spades.py -1 sub_read1.fastq -2 sub_read2.fastq -o spades_sub_output > /dev/null
seqkit stat -a spades_output/scaffolds.fasta spades_sub_output/scaffolds.fasta

### 考えてみよう
- リードの量を減らすと、配列数・全長・N50はどう変わりましたか？
- 全長が短くなった場合、ゲノムのどの部分が復元できなくなったと考えられますか？
- 実際のゲノム解読では、目的に応じてどのくらいのカバレッジのリードを用意すればよいか、費用とのバランスで決めます。どのような点を考える必要があるでしょうか？

---
## まとめ

　今回、ショートリードとロングリードを使ってゲノムアセンブルをおこない、ゲノム配列の復元を試みました。

　ショートリードでは、正確なコンティグを得られる一方で、ゲノム全体を再現するほど充分長い配列は得られませんでした。要因としては、ゲノムに含まれるリピート配列の存在などが考えられます。

　ロングリードでは、非常に長いコンティグを得ることができました。しかし、エラー率が高いリードを使ったため、部分的には塩基情報が不正確なコンティグが生成されました。

　ショートリードとロングリードそれぞれの利点を活かすことで、より良いゲノム配列を構築することができます。今回の実習でも、ロングリードで得られた長いコンティグを、ショートリードの正確性で補い、高精度なゲノム配列を得ることができました。このようなロングリードとショートリードを組み合わせたゲノム解読は、現在よく使われています。

　また、アセンブリの良し悪しは、N50のような長さの指標だけでは判断できないこともわかりました。長いコンティグであっても、つなぎ間違いが含まれていることがあります。アセンブリを評価するときは、長さと正確さの両方を確かめることが大切です。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/Assemble_figs/major_assemble_proc.png?raw=true" height="220px" align="middle">

<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>

#### ソフトウェアのインストールがうまくいかない場合
　次のコードセルを実行して、別の方法でのインストールを試してみてください。

In [ ]:
### ソフトウェアのインストール
## igv viewer
!pip install -q igv-notebook
!wget -q -O igv_prep.py https://github.com/CropEvol/lecture/raw/master/textbook_2025/scripts/igv_prep.py
## conda-env
!pip install -q condacolab
import condacolab
condacolab.install_miniforge()
## softwares
!conda install -q -y bioconda::seqkit
!conda install -q -y bioconda::spades
!conda install -q -y bioconda::flye
!conda install -q -y bioconda::minimap2
!conda install -q -y bioconda::samtools
!conda install -q -y bioconda::bwa
!conda install -q -y bioconda::pilon
## scripts
!wget -q -O Alignment_mm2.sh https://github.com/CropEvol/lecture/raw/master/textbook_2025/scripts/Alignment_mm2.sh
!wget -q -O Polishing.sh     https://github.com/CropEvol/lecture/raw/master/textbook_2025/scripts/Polishing.sh

#### このテキストで出力される結果ファイル


In [ ]:
## ショートリードアセンブリ
!mkdir -p spades_output && wget -q -O spades_output/scaffolds.fasta https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/Genome_assemble_output/spades_scaffolds.fasta
## ロングリードアセンブリ
!mkdir -p flye_output && wget -q -O flye_output/assembly.fasta https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/Genome_assemble_output/flye_assembly.fasta
## 補正後のアセンブリ
!wget -q -O assembly_polished.fasta https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/Genome_assemble_output/assembly_polished.fasta
## IGV用データ
!wget -q -O sample_genome.fa https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/sample_genome.fa
!wget -q -O sample_genome.fa.fai https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/Genome_assemble_output/sample_genome.fa.fai
!wget -q -O ShortReadAssembly.bam https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/Genome_assemble_output/ShortReadAssembly.bam
!wget -q -O LongReadAssembly.bam https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/Genome_assemble_output/LongReadAssembly.bam
!wget -q -O LongReadAssembly_polished.bam https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/Genome_assemble_output/LongReadAssembly_polished.bam